# Project 13 · BERT Baseline for Disease Named Entity Recognition

Fine-tune `bert-base-cased` on NCBI Disease as the general-purpose reference for the biomedical models.

In [1]:
%pip install "datasets==3.6.0"

  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.19
    Uninstalling multiprocess-0.70.19:
      Successfully uninstalled multiprocess-0.70.19
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [datasets]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


## 1. Set up the environment

Use a Jupyter kernel with PyTorch, Transformers, Datasets, Evaluate, and seqeval. This notebook records Datasets `3.6.0` for the NCBI Disease loader.

In [ ]:
!pip install torch datasets transformers evaluate seqeval

## 2. Load and inspect the dataset

Load the NCBI Disease dataset with its existing training, validation, and test splits. Inspect a training sentence and the three BIO labels: `O`, `B-Disease`, and `I-Disease`.

In [2]:
from datasets import load_dataset

dataset = load_dataset("ncbi/ncbi_disease")

print(dataset)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'TypeError: Failed to fetch'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/9.70k [00:00<?, ?B/s]

ncbi_disease.py:   0%|          | 0.00/5.83k [00:00<?, ?B/s]

The repository for ncbi/ncbi_disease contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/ncbi/ncbi_disease.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/5433 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/924 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/941 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 5433
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 924
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 941
    })
})


In [3]:
from audit_ncbi_splits import audit_ncbi_splits

audit_ncbi_splits(dataset)


train: DIFFERENT
  Sentences: Hugging Face=5433, CoNLL=5432
  Rows with different tokens: 0
  Rows with different BIO labels: 0
  Rows present in only one source: 1
  Hugging Face BIO counts: {'O': 124819, 'B-Disease': 5145, 'I-Disease': 6122}
  Annotated disease starts (B-Disease): 5145
  Orphan I-Disease tags: 0
  CoNLL SHA-256: 5f87b0de649eccff21e6d6e49e304aace808056ecae55225ea31602192517be9
  First difference at zero-based row 5432:
    Hugging Face: ([], [])
    CoNLL: None

validation: DIFFERENT
  Sentences: Hugging Face=924, CoNLL=923
  Rows with different tokens: 0
  Rows with different BIO labels: 0
  Rows present in only one source: 1
  Hugging Face BIO counts: {'O': 22092, 'B-Disease': 787, 'I-Disease': 1090}
  Annotated disease starts (B-Disease): 787
  Orphan I-Disease tags: 0
  CoNLL SHA-256: b4e1e7efcc2a35047bf2f3d281648357470e674f555d4a26a0049ac37d59cc61
  First difference at zero-based row 923:
    Hugging Face: ([], [])
    CoNLL: None

test: DIFFERENT
  Sentences: H

False

In [ ]:
%pip install "datasets==3.6.0"

Note: you may need to restart the kernel to use updated packages.


In [13]:
example = dataset["train"][0]

print("ID:")
print(example["id"])

print("\nTokens:")
print(example["tokens"])

print("\nNER tags:")
print(example["ner_tags"])

ID:
0

Tokens:
['Identification', 'of', 'APC2', ',', 'a', 'homologue', 'of', 'the', 'adenomatous', 'polyposis', 'coli', 'tumour', 'suppressor', '.']

NER tags:
[0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0]


In [ ]:
label_names = dataset["train"].features["ner_tags"].feature.names

print(label_names)

['O', 'B-Disease', 'I-Disease']


In [ ]:
for token, tag in zip(example["tokens"], example["ner_tags"]):
    print(f"{token:20} {label_names[tag]}")

Identification       O
of                   O
APC2                 O
,                    O
a                    O
homologue            O
of                   O
the                  O
adenomatous          B-Disease
polyposis            I-Disease
coli                 I-Disease
tumour               I-Disease
suppressor           O
.                    O


## 3. Tokenize and align labels

Assign the original BIO label to the first subword of each word. Use `-100` for later subwords and special tokens so they are ignored. Inspect one example before applying the same alignment to all splits.

In [7]:
from transformers import AutoTokenizer

model_name = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print(tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [8]:
limit = tokenizer.model_max_length

for split, rows in nonempty_dataset.items():
    encoded = tokenizer(
        rows["tokens"],
        is_split_into_words=True,
        truncation=False,
        padding=False,
    )
    lengths = [len(ids) for ids in encoded["input_ids"]]
    over_limit = sum(length > limit for length in lengths)

    print(f"{split}: longest = {max(lengths)}, over {limit} = {over_limit}/{len(lengths)}")

train: longest = 178, over 512 = 0/5432
validation: longest = 122, over 512 = 0/923
test: longest = 157, over 512 = 0/940


In [9]:
from transformers import AutoTokenizer

pubmed_tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract",
    use_fast=True,
    do_lower_case=True,
)

limit = 512  # 与组里的 PubMedBERT notebook 一致

for split, rows in nonempty_dataset.items():
    encoded = pubmed_tokenizer(
        rows["tokens"],
        is_split_into_words=True,
        truncation=False,
        padding=False,
    )
    lengths = [len(ids) for ids in encoded["input_ids"]]
    over_limit = sum(length > limit for length in lengths)

    print(f"{split}: longest = {max(lengths)}, over {limit} = {over_limit}/{len(lengths)}")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

train: longest = 139, over 512 = 0/5432
validation: longest = 95, over 512 = 0/923
test: longest = 130, over 512 = 0/940


In [10]:
biobert_tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2",
    revision="67c9c25b46986521ca33df05d8540da1210b3256",
    use_fast=True,
    do_lower_case=False,
)

for split, rows in nonempty_dataset.items():
    encoded = biobert_tokenizer(
        rows["tokens"],
        is_split_into_words=True,
        truncation=False,
        padding=False,
    )
    lengths = [len(ids) for ids in encoded["input_ids"]]
    over_limit = sum(length > 512 for length in lengths)
    print(f"{split}: longest = {max(lengths)}, over 512 = {over_limit}/{len(lengths)}")

config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

train: longest = 178, over 512 = 0/5432
validation: longest = 122, over 512 = 0/923
test: longest = 157, over 512 = 0/940


### Tokenized input lengths

We measured sentence lengths after tokenization, including special tokens,
with truncation disabled. No sentence exceeded the 512-token limit.

| Tokenizer | Longest train input | Longest validation input | Longest test input |
|---|---:|---:|---:|
| BERT (`bert-base-cased`) | 178 | 122 | 157 |
| PubMedBERT | 139 | 95 | 130 |
| BioBERT v1.2 | 178 | 122 | 157 |

In [14]:
tokens = example["tokens"]

tokenized = tokenizer(
    tokens,
    is_split_into_words=True
)

bert_tokens = tokenizer.convert_ids_to_tokens(tokenized["input_ids"])

print(bert_tokens)

['[CLS]', 'I', '##dent', '##ification', 'of', 'AP', '##C', '##2', ',', 'a', 'ho', '##mo', '##logue', 'of', 'the', 'ad', '##eno', '##mat', '##ous', 'p', '##oly', '##po', '##sis', 'co', '##li', 't', '##umour', 'suppress', '##or', '.', '[SEP]']


In [15]:
word_ids = tokenized.word_ids()

for bert_token, word_id in zip(bert_tokens, word_ids):
    print(f"{bert_token:15} {word_id}")

[CLS]           None
I               0
##dent          0
##ification     0
of              1
AP              2
##C             2
##2             2
,               3
a               4
ho              5
##mo            5
##logue         5
of              6
the             7
ad              8
##eno           8
##mat           8
##ous           8
p               9
##oly           9
##po            9
##sis           9
co              10
##li            10
t               11
##umour         11
suppress        12
##or            12
.               13
[SEP]           None


In [16]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        label_ids = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)

            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

In [18]:
label_names = dataset["train"].features["ner_tags"].feature.names

In [19]:
test_data = dataset["train"].select([0])

test_result = tokenize_and_align_labels(test_data)

test_tokens = tokenizer.convert_ids_to_tokens(
    test_result["input_ids"][0]
)

test_labels = test_result["labels"][0]

for token, label in zip(test_tokens, test_labels):
    if label == -100:
        label_name = "IGNORE"
    else:
        label_name = label_names[label]

    print(f"{token:15} {label_name}")

[CLS]           IGNORE
I               O
##dent          IGNORE
##ification     IGNORE
of              O
AP              O
##C             IGNORE
##2             IGNORE
,               O
a               O
ho              O
##mo            IGNORE
##logue         IGNORE
of              O
the             O
ad              B-Disease
##eno           IGNORE
##mat           IGNORE
##ous           IGNORE
p               I-Disease
##oly           IGNORE
##po            IGNORE
##sis           IGNORE
co              I-Disease
##li            IGNORE
t               I-Disease
##umour         IGNORE
suppress        O
##or            IGNORE
.               O
[SEP]           IGNORE


### BIO label alignment example

The annotated disease mention is `adenomatous polyposis coli tumour`.
BERT splits `adenomatous` into `ad`, `##eno`, `##mat`, and `##ous`.
The first subword, `ad`, keeps the original `B-Disease` label.
The remaining subwords receive `-100`, shown as `IGNORE`.
The first subword of each of the next three words keeps its `I-Disease` label.
Special tokens such as `[CLS]` and `[SEP]` also receive `-100`.
Positions labelled `-100` are excluded from the training loss and evaluation.

In [20]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

print(tokenized_dataset)

Map:   0%|          | 0/5433 [00:00<?, ? examples/s]

Map:   0%|          | 0/924 [00:00<?, ? examples/s]

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5433
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 924
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 941
    })
})


## 4. Train, validate, and evaluate

Fine-tune for three epochs with a learning rate of `2e-5`, a per-device batch size of `8`, and weight decay of `0.01`. Evaluate on the validation split after each epoch. Report entity-level precision, recall, and F1 on the test split, alongside word-level label accuracy.

In [ ]:
from transformers import AutoModelForTokenClassification

id2label = {
    0: "O",
    1: "B-Disease",
    2: "I-Disease"
}

label2id = {
    "O": 0,
    "B-Disease": 1,
    "I-Disease": 2
}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print(model.__class__.__name__)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11293.97it/s]
[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not

BertForTokenClassification


In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_names[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_names[l] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert-ner-baseline",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
train_result = trainer.train()

/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.041484,0.051802,0.777647,0.839898,0.807575,0.983896
2,0.025029,0.051599,0.770455,0.861499,0.813437,0.984522
3,0.023125,0.060924,0.798824,0.862770,0.829566,0.985523


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.83it/s]
/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.12it/s]
/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.79it/s]


In [ ]:
test_results = trainer.evaluate(tokenized_dataset["test"])

print("Precision:", test_results["eval_precision"])
print("Recall:", test_results["eval_recall"])
print("F1:", test_results["eval_f1"])
print("Accuracy:", test_results["eval_accuracy"])

/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.023125,0.066226,3,0.826172,0.881250,0.852823,0.983876


Precision: 0.826171875
Recall: 0.88125
F1: 0.8528225806451613
Accuracy: 0.9838755766012165


## 5. Analyze validation errors

Run this section with the trained baseline `trainer`, after the training cell has finished. This section compares predictions with gold labels on the validation split. Span positions use zero-based token indices with an exclusive end. A sentence can contain more than one error type. A boundary error counts once per affected gold entity; the other counts cover unmatched spans with no overlap.


In [ ]:
from collections import Counter

import pandas as pd
from IPython.display import display
from seqeval.metrics.sequence_labeling import get_entities


def validation_errors(examples, predictions, references, label_names):
    if len(examples) != len(predictions) or len(examples) != len(references):
        raise ValueError("Expected one prediction and reference per validation example.")

    counts = Counter({"Missed entity": 0, "False positive": 0, "Boundary error": 0})
    cases = []
    truncated = 0

    for row_index, (example, predicted_ids, reference_ids) in enumerate(
        zip(examples, predictions, references)
    ):
        kept = reference_ids != -100
        word_predictions = predicted_ids[kept]
        word_references = reference_ids[kept]
        tokens = example["tokens"]

        if word_references.tolist() != example["ner_tags"][: len(word_references)]:
            raise ValueError(f"Word labels do not align for validation row {row_index}.")
        if len(word_references) != len(tokens):
            # A truncated sequence cannot be compared against the entire sentence.
            truncated += 1
            continue

        gold = set(get_entities([label_names[i] for i in word_references]))
        predicted = set(get_entities([label_names[i] for i in word_predictions]))
        missed = gold - predicted
        extra = predicted - gold

        # Overlapping spans with different endpoints count as boundary errors.
        # Count each affected gold span once, even if multiple predictions overlap it.
        boundary = set()
        overlapping_predictions = set()
        for reference in missed:
            _, start, end = reference
            for prediction in extra:
                _, predicted_start, predicted_end = prediction
                if start <= predicted_end and predicted_start <= end:
                    boundary.add(reference)
                    overlapping_predictions.add(prediction)

        errors = {
            "Missed entity": len(missed - boundary),
            "False positive": len(extra - overlapping_predictions),
            "Boundary error": len(boundary),
        }
        counts.update(errors)
        if any(errors.values()):
            def entity_text(spans):
                return ", ".join(
                    f"{' '.join(tokens[start:end + 1])} [{start}:{end + 1}]"
                    for _, start, end in sorted(spans, key=lambda span: span[1:])
                ) or "—"

            cases.append({
                "Validation row": row_index,
                "Sentence": " ".join(tokens),
                "Gold disease spans": entity_text(gold),
                "Predicted disease spans": entity_text(predicted),
                "Error types": ", ".join(kind for kind, count in errors.items() if count),
            })

    return counts, cases, truncated


validation_output = trainer.predict(tokenized_dataset["validation"])
predicted_ids = np.argmax(validation_output.predictions, axis=-1)
error_counts, error_cases, truncated_rows = validation_errors(
    dataset["validation"], predicted_ids, validation_output.label_ids, label_names
)

print(f"Validation rows analysed: {len(predicted_ids) - truncated_rows}")
print(f"Truncated rows excluded from error counts: {truncated_rows}")
print(f"Rows with at least one error: {len(error_cases)}")
display(pd.DataFrame(error_counts.items(), columns=["Error type", "Count"]))

# Show one example of each available error type, up to five distinct sentences.
selected_cases = {}
for error_type in error_counts:
    for case in error_cases:
        if error_type in case["Error types"].split(", "):
            selected_cases[case["Validation row"]] = case
            break
for case in error_cases:
    if len(selected_cases) >= 5:
        break
    selected_cases[case["Validation row"]] = case

if selected_cases:
    with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
        display(pd.DataFrame(selected_cases.values()))
else:
    print("No entity errors found in the analysed rows.")

In [4]:
nonempty_dataset = dataset.filter(lambda row: bool(row["tokens"]))
audit_ncbi_splits(nonempty_dataset)

Filter:   0%|          | 0/5433 [00:00<?, ? examples/s]

Filter:   0%|          | 0/924 [00:00<?, ? examples/s]

Filter:   0%|          | 0/941 [00:00<?, ? examples/s]


train: EXACT MATCH
  Sentences: Hugging Face=5432, CoNLL=5432
  Rows with different tokens: 0
  Rows with different BIO labels: 0
  Rows present in only one source: 0
  Hugging Face BIO counts: {'O': 124819, 'B-Disease': 5145, 'I-Disease': 6122}
  Annotated disease starts (B-Disease): 5145
  Orphan I-Disease tags: 0
  CoNLL SHA-256: 5f87b0de649eccff21e6d6e49e304aace808056ecae55225ea31602192517be9

validation: EXACT MATCH
  Sentences: Hugging Face=923, CoNLL=923
  Rows with different tokens: 0
  Rows with different BIO labels: 0
  Rows present in only one source: 0
  Hugging Face BIO counts: {'O': 22092, 'B-Disease': 787, 'I-Disease': 1090}
  Annotated disease starts (B-Disease): 787
  Orphan I-Disease tags: 0
  CoNLL SHA-256: b4e1e7efcc2a35047bf2f3d281648357470e674f555d4a26a0049ac37d59cc61

test: EXACT MATCH
  Sentences: Hugging Face=940, CoNLL=940
  Rows with different tokens: 0
  Rows with different BIO labels: 0
  Rows present in only one source: 0
  Hugging Face BIO counts: {'O': 

True

### Dataset source and split verification

We use the NCBI Disease Corpus for disease named entity recognition.
The dataset uses three BIO labels: `O`, `B-Disease`, and `I-Disease`.

The Hugging Face loader produced one empty record at the end of each split.
After excluding these records, we compared the Hugging Face and CoNLL
versions sentence by sentence. The sentence order, tokens, and BIO labels
matched exactly in all three splits.

| Split | Non-empty sentences | Annotated disease mentions |
|---|---:|---:|
| Train | 5,432 | 5,145 |
| Validation | 923 | 787 |
| Test | 940 | 960 |

In [6]:
from collections import Counter
import pandas as pd

summary = {}

for split, rows in nonempty_dataset.items():
    counts = Counter()

    for row in rows:
        length = 0

        for tag in row["ner_tags"] + [0]:
            if tag == 2:  # I-Disease
                length += 1
            else:
                if length:
                    counts[min(length, 4)] += 1
                length = 1 if tag == 1 else 0  # B-Disease

    summary[split] = {
        "1 word": counts[1],
        "2 words": counts[2],
        "3 words": counts[3],
        "4+ words": counts[4],
        "Total": sum(counts.values()),
    }

pd.DataFrame.from_dict(summary, orient="index")

,1 word,2 words,3 words,4+ words,Total
train,2266,1402,643,834,5145
validation,283,225,122,157,787
test,423,270,133,134,960


### Disease mention lengths

We counted the number of original words in each annotated disease mention
before splitting words into subword tokens.

Multi-word mentions account for 56.0% of training mentions, 64.0% of
validation mentions, and 55.9% of test mentions. Since more than half of
the mentions span multiple words in every split, identifying the full
span is important. For example, predicting only `cancer` when the
annotated mention is `breast cancer` misses the full entity boundary.